# Notebook 02 — Stochastic Simulation: Gillespie Algorithm

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 2 of 12  
**Time estimate:** 75 minutes

> The SIR ODE assumes millions of identical, well-mixed particles. A single
> cell has ~10 molecules of a transcription factor — at this scale, randomness
> is the signal, not the noise. The Gillespie algorithm simulates *exact*
> stochastic chemical kinetics, one reaction at a time.

---
## Step 1 — Motivation

Elowitz et al. (2002, Science) measured the expression of two identical genes
in the same *E. coli* cell and found they expressed at different levels — not
because the genes were different, but because the *timing* of individual
transcription and translation events is random. This noise is intrinsic
(stochastic firing of molecular reactions) and cannot be averaged away — it
is why genetically identical cells in the same environment can have different
fates. ODEs cannot capture this; Gillespie simulation can.

---
## Step 2 — Intuition

The **Gillespie SSA (Stochastic Simulation Algorithm)** simulates a well-mixed
chemical system with integer molecule counts.

**Core idea:** at any moment, each reaction fires at a rate proportional to
how many molecules can currently react (the *propensity*). The time until the
next reaction is exponentially distributed with rate = sum of all propensities.
Which reaction fires is chosen proportional to its propensity.

**Two questions the algorithm answers at every step:**
1. *When* does the next reaction fire? → exponential waiting time
2. *Which* reaction fires? → proportional to propensities

**Intrinsic vs. extrinsic noise:**
- **Intrinsic noise:** randomness in reaction timing for a specific gene/protein
  (Gillespie captures this)
- **Extrinsic noise:** cell-to-cell variation in the environment (RNA polymerase
  abundance, cell size) — requires multiple simulated cells with different parameters

---
## Step 3 — Biological Background

**Gene expression as a stochastic process:**
- Transcription: RNA polymerase finds and binds the promoter at rate $k_{tx}$;
  mRNA degrades at rate $\gamma_m$.
- Translation: ribosome binds mRNA and produces protein at rate $k_{tl}$;
  protein degrades at rate $\gamma_p$.
- At steady state: $\langle m \rangle = k_{tx}/\gamma_m$,
  $\langle p \rangle = k_{tl}\langle m \rangle / \gamma_p$
- But the variance scales with the mean (Fano factor > 1 in bursty expression).

**Relevance:**
- Cell fate decisions (stem cell differentiation, apoptosis) can be driven by
  stochastic noise crossing a threshold — not a deterministic signal.
- Drug dosing: stochastic model shows that even with average drug concentration
  suppressing a pathogen, a fraction of cells escape due to noise.

**Birth-death process:** the simplest stochastic system — particles are created
at rate $\lambda$ and destroyed at rate $\mu$ per particle. The Gillespie SSA
reproduces this exactly, and the stationary distribution is Poisson$(\lambda/\mu)$.

---
## Step 4 — Mathematical Explanation

**Propensity functions:** for a reaction with rate constant $c_j$ and reactant
molecule counts, the propensity $a_j$ is the probability per unit time that
reaction $j$ fires:
- Zeroth order (birth, $\emptyset \to X$): $a = c$
- First order ($X \to \emptyset$): $a = c \cdot N_X$
- Second order ($X + Y \to \ldots$): $a = c \cdot N_X \cdot N_Y$

**Gillespie SSA (direct method):**
1. Compute all propensities $a_j(t)$ and total $a_0 = \sum_j a_j$
2. Time to next reaction: $\tau \sim \text{Exp}(a_0)$, i.e., $\tau = -\ln(u_1)/a_0$, $u_1 \sim U(0,1)$
3. Which reaction: choose $\mu$ such that $\sum_{j=1}^{\mu-1} a_j < u_2 a_0 \leq \sum_{j=1}^{\mu} a_j$, $u_2 \sim U(0,1)$
4. Update state: apply reaction $\mu$'s stoichiometry
5. Advance time: $t \leftarrow t + \tau$
6. Repeat until $t > t_{\max}$

**Tau-leaping (approximate):** instead of simulating one reaction at a time,
fire each reaction $k_j \sim \text{Poisson}(a_j \cdot \tau)$ times in a leap $\tau$.
Faster but approximate; accuracy requires $a_j \tau \ll a_j$ (leap condition).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

rng = np.random.default_rng(42)

# ---- Gillespie SSA — general implementation ----
def gillespie(propensity_fn, update_fn, x0, t_max, rng):
    """
    General Gillespie SSA.
    
    Args:
        propensity_fn: f(x) -> list of propensities for each reaction
        update_fn: f(x, reaction_idx) -> new state x after reaction fires
        x0: initial state (np.array of molecule counts)
        t_max: float
        rng: np.random.Generator
    Returns:
        times: list of floats
        states: list of np.arrays
    """
    t = 0.0
    x = np.array(x0, dtype=float)
    times  = [t]
    states = [x.copy()]

    while t < t_max:
        a = propensity_fn(x)
        a0 = sum(a)
        if a0 == 0:
            break  # absorbing state
        # Time to next reaction
        tau = -np.log(rng.random()) / a0
        # Which reaction fires
        u2 = rng.random() * a0
        cumsum = 0
        mu = 0
        for j, aj in enumerate(a):
            cumsum += aj
            if cumsum >= u2:
                mu = j
                break
        # Update state
        x = update_fn(x, mu)
        t += tau
        times.append(t)
        states.append(x.copy())

    return np.array(times), np.array(states)

# ---- System 1: Birth-death process ----
# Reaction 0: birth  ∅ → X, rate λ
# Reaction 1: death  X → ∅, rate μ per molecule
lam, mu_bd = 5.0, 0.5  # steady state = λ/μ = 10

def bd_propensity(x):
    return [lam, mu_bd * x[0]]

def bd_update(x, mu):
    x = x.copy()
    if mu == 0: x[0] += 1
    else:       x[0] = max(0, x[0] - 1)
    return x

# Run 50 trajectories
N_TRAJ = 50
T_MAX = 50.0
trajectories_bd = []
for _ in range(N_TRAJ):
    t_bd, s_bd = gillespie(bd_propensity, bd_update, [0], T_MAX, rng)
    trajectories_bd.append((t_bd, s_bd[:, 0]))

print(f'Birth-death: steady-state mean = λ/μ = {lam/mu_bd:.1f}')
print(f'Simulated mean (t>40): {np.mean([np.mean(s[t > 40]) for t, s in trajectories_bd]):.2f}')

# ---- System 2: Gene expression (mRNA + protein) ----
# State: [mRNA, protein]
# Reactions:
#   0: transcription ∅ → M, rate k_tx
#   1: translation   M → M + P, rate k_tl per mRNA
#   2: mRNA decay    M → ∅, rate gamma_m per mRNA
#   3: protein decay P → ∅, rate gamma_p per protein
k_tx, k_tl, gamma_m, gamma_p = 1.0, 5.0, 0.2, 0.1
ss_mRNA    = k_tx / gamma_m       # = 5
ss_protein = k_tl * ss_mRNA / gamma_p  # = 250
print(f'\nGene expression: mRNA SS={ss_mRNA:.0f}, protein SS={ss_protein:.0f}')

def ge_propensity(x):
    m, p = x
    return [k_tx, k_tl * m, gamma_m * m, gamma_p * p]

def ge_update(x, mu):
    x = x.copy()
    if   mu == 0: x[0] += 1       # transcription
    elif mu == 1: x[1] += 1       # translation
    elif mu == 2: x[0] = max(0, x[0] - 1)  # mRNA decay
    elif mu == 3: x[1] = max(0, x[1] - 1)  # protein decay
    return x

# Run 5 trajectories of gene expression (protein)
T_GE = 200.0
ge_trajs = []
for _ in range(5):
    t_ge, s_ge = gillespie(ge_propensity, ge_update, [0, 0], T_GE, rng)
    ge_trajs.append((t_ge, s_ge))
print(f'Gene expression runs complete.')

In [ ]:
# ---- Tau-leaping (approximate SSA) ----
def tau_leap(propensity_fn, stoich, x0, t_max, tau_step, rng):
    """
    Explicit tau-leaping: fire Poisson(a_j * tau) copies of each reaction per step.
    stoich: (n_reactions, n_species) stoichiometry matrix
    """
    t = 0.0
    x = np.array(x0, dtype=float)
    times = [t]
    states = [x.copy()]
    while t < t_max:
        a = np.array(propensity_fn(x))
        # Number of firings of each reaction in [t, t+tau]
        k = rng.poisson(a * tau_step)
        dx = stoich.T @ k
        x = np.maximum(0, x + dx)  # clamp to 0
        t += tau_step
        times.append(t)
        states.append(x.copy())
    return np.array(times), np.array(states)

# Birth-death stoichiometry: row = reaction, col = species
bd_stoich = np.array([[1], [-1]])  # birth: +1; death: -1

t_tl, s_tl = tau_leap(bd_propensity, bd_stoich, [0], T_MAX, tau_step=0.2, rng=rng)
print(f'Tau-leaping mean (t>40): {np.mean(s_tl[int(40/0.2):, 0]):.2f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel A: Birth-death trajectories + stationary distribution
ax = axes[0, 0]
for t_bd, s_bd in trajectories_bd[:10]:
    ax.step(t_bd, s_bd, where='post', alpha=0.3, color='steelblue', lw=0.8)
ax.axhline(lam/mu_bd, color='tomato', lw=2, ls='--', label=f'Steady state (λ/μ={lam/mu_bd:.0f})')
ax.set_xlabel('Time'); ax.set_ylabel('Molecule count X')
ax.set_title('A. Birth-death process\n(10 Gillespie trajectories)')
ax.legend(fontsize=8)

# Panel B: Stationary distribution vs. Poisson(λ/μ)
ax = axes[0, 1]
final_counts = [s[-1] for _, s in trajectories_bd]
counts_range = range(0, 30)
ax.bar(counts_range,
         [sum(1 for c in final_counts if c == n) / N_TRAJ for n in counts_range],
         alpha=0.7, color='steelblue', label='Simulated')
k_vals = np.array(counts_range)
ax.plot(k_vals, poisson.pmf(k_vals, lam/mu_bd), 'ro-', ms=5, lw=1.5,
          label=f'Poisson(λ/μ={lam/mu_bd:.0f})')
ax.set_xlabel('Count'); ax.set_ylabel('Probability')
ax.set_title('B. Stationary distribution\nvs. Poisson prediction')
ax.legend(fontsize=8)

# Panel C: Gene expression — protein trajectories
ax = axes[0, 2]
colors = ['steelblue','tomato','green','orange','purple']
for i, (t_ge, s_ge) in enumerate(ge_trajs):
    ax.step(t_ge, s_ge[:, 1], where='post', alpha=0.7,
              color=colors[i], lw=1, label=f'Cell {i+1}')
ax.axhline(ss_protein, color='k', ls='--', lw=1.5, label=f'ODE SS ({ss_protein:.0f})')
ax.set_xlabel('Time'); ax.set_ylabel('Protein count')
ax.set_title('C. Gene expression noise\n(5 identical cells, same parameters)')
ax.legend(fontsize=7)

# Panel D: mRNA and protein for one cell
ax = axes[1, 0]
t_ge, s_ge = ge_trajs[0]
ax2 = ax.twinx()
ax.step(t_ge, s_ge[:, 0], where='post', color='steelblue', lw=1.5, label='mRNA')
ax2.step(t_ge, s_ge[:, 1], where='post', color='tomato', lw=1.5, label='Protein')
ax.set_xlabel('Time'); ax.set_ylabel('mRNA count', color='steelblue')
ax2.set_ylabel('Protein count', color='tomato')
ax.set_title('D. mRNA and protein dynamics\n(one cell, gene expression SSA)')
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=9)

# Panel E: Gillespie vs tau-leaping comparison
ax = axes[1, 1]
# Run a single Gillespie for comparison
t_ssa, s_ssa = gillespie(bd_propensity, bd_update, [0], 50, rng)
ax.step(t_ssa, s_ssa[:, 0], where='post', color='steelblue', lw=1.5, label='Gillespie (exact)', alpha=0.9)
ax.step(t_tl,  s_tl[:, 0],  where='post', color='tomato', lw=1.5, label='Tau-leaping (τ=0.2)', alpha=0.9)
ax.axhline(lam/mu_bd, color='k', ls='--', lw=1)
ax.set_xlabel('Time'); ax.set_ylabel('Count')
ax.set_title('E. Gillespie vs. tau-leaping\n(birth-death process)')
ax.legend(fontsize=9)

# Panel F: Noise vs. molecule count
ax = axes[1, 2]
mu_vals = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
fano_factors = []
for mu_v in mu_vals:
    # SS mean = lam/mu_v
    traj_counts = []
    for _ in range(100):
        t_v, s_v = gillespie(
            lambda x, m=mu_v: [lam, m * x[0]],
            bd_update, [0], 100, rng)
        # Collect counts at t > 50
        late = s_v[t_v > 50, 0]
        if len(late) > 0:
            traj_counts.extend(late)
    fano_factors.append(np.var(traj_counts) / max(np.mean(traj_counts), 1e-9))
ss_means = [lam/m for m in mu_vals]
ax.semilogx(ss_means, fano_factors, 'o-', color='steelblue', lw=2)
ax.axhline(1.0, color='k', ls='--', lw=1, label='Poisson (Fano=1)')
ax.set_xlabel('Steady-state mean molecule count')
ax.set_ylabel('Fano factor (Var/Mean)')
ax.set_title('F. Fano factor vs. molecule count\n(intrinsic noise dominates at low N)')
ax.legend(fontsize=9)

plt.suptitle('Module 15 NB02: Stochastic Simulation — Gillespie Algorithm', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('stochastic_simulation_gillespie.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement tau-leaping for the gene expression system. Compare simulation
   speed (number of steps) vs. Gillespie at τ=0.1, 0.5, 1.0. At which τ
   does the protein distribution deviate from the exact Gillespie result?
2. The birth-death stationary distribution is Poisson(λ/μ) with Fano factor = 1.
   Add bursty transcription: instead of firing one mRNA per transcription event,
   fire a Geometric(1/b) burst with mean burst size b. How does the Fano factor
   change? (Expected answer: Fano = 1 + b.)
3. Implement the stochastic SIR model using Gillespie (same β, γ as NB01).
   Run 20 trajectories with N=1000. Compare to the ODE SIR solution. What
   fraction of trajectories go extinct before the epidemic takes off?

---
## Step 10 — Quiz

1. What is a propensity function? Write the propensity for a second-order
   reaction A + B → C with rate constant c.
2. What two random numbers does the Gillespie SSA draw per step, and what
   do they determine?
3. What is the difference between intrinsic and extrinsic noise in gene
   expression? Which does the Gillespie SSA model?
4. What is the leap condition in tau-leaping, and why must it be satisfied
   for tau-leaping to be valid?

---
## Step 12 — Reflection

> *[In one paragraph: explain why a Gillespie simulation of gene expression
> with mean 5 mRNA molecules produces much more variable protein output than
> an ODE predicts. Use the Fano factor concept. Why does this matter for
> understanding cell-to-cell variability in a clonal bacterial population?]*

---
*Next: `03_reaction_diffusion_systems.ipynb`*